# Лабораторна 3. Токенізація та трансформери

У цій лабораторній роботі ви пройдете шлях від механіки токенізації до практичного використання готових мовних моделей:

- алгоритми токенізації (BPE, WordPiece, Unigram) — натренуєте всі три на власному тексті;
- mini-GPT з нуля — повторите тренування трансформера на своєму датасеті;
- готові моделі для різних NLP-задач — класифікація, генерація, переклад, Q&A та інше через `pipeline()`;
- порівняння кількох моделей на одній задачі;
- творчий застосунок на основі готової моделі.

Демо-ноутбук лекції 4 (`ransformers_demo.ipynb`) — основа для секцій 2-3, звертайтесь до нього як до довідника. Розділи 1 і 4 спираються на офіційний [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/en/chapter1/1) — читайте відповідні розділи курсу перед виконанням, посилання додані в кожній секції.

## Матеріали

- [3Blue1Brown — Transformers, the tech behind LLMs](https://www.youtube.com/watch?v=wjZofJX0v4M)
- [3Blue1Brown — Attention in transformers, step-by-step](https://www.youtube.com/watch?v=eMlx5fFNoYc)
- [3Blue1Brown — How might LLMs store facts](https://www.youtube.com/watch?v=9-Jl0dxWQs8)
- [Hugging Face LLM Course — Chapter 1: Transformer Models](https://huggingface.co/learn/llm-course/en/chapter1/1)
- [Hugging Face LLM Course — Chapter 2: Using Transformers](https://huggingface.co/learn/llm-course/en/chapter2/1)
- [Hugging Face LLM Course — Chapter 6: The Tokenizers library](https://huggingface.co/learn/llm-course/en/chapter6/1)

## 0. Опрацюйте демо-ноутбук лекції 4

Прогляньте пояснення і запустіть усі клітинки демо-ноутбука.

Після цього дайте відповідь на запитання нижче.

### 0.1. Теоретичні питання (якщо потрібно дописати якийсь код, то в блоці "Мої відповіді наведіть відповідний код")

1. У розділі 2.1 демо тренер BPE отримує список окремих віршів, а не один склеєний рядок. Що б змінилось (і що б зламалось), якби ми передали один суцільний рядок?
2. З який етапів складається токенізація, поясніть сутність кожного з етапів (див. [туторіал по токенізації](https://huggingface.co/learn/llm-course/en/chapter6/8))
2. Що таке `"<unk>", "<eos>"` в 2.1.?
3. Зменшіть `VOCAB_SIZE` (2.1) удвічі-втричі (наприклад з 16000 до 4000-6000) і перетренуйте BPE-токенізатор на тому самому корпусі. Порівняйте середню кількість токенів на вірш і частку `<unk>` до/після зміни. Поясніть, чому менший словник не підвищує `<unk>`, але впливає на довжину закодованих послідовностей (а отже — і на кількість кроків, які "бачить" модель за той самий `CONTEXT_LEN`).
4. У розділі 2.2 між віршами вставлено токен `<eos>`. Навіщо? Що конкретно піде не так під час тренування mini-GPT, якщо цей роздільник прибрати?
5. Який формат мають і що репрезентують елементи тензорів `train_data` & `val_data` у розділі 2.3?
6. Допишіть в 3.2. `assert` тест, який перевіряє відповідність частин випадкового батчу.
7. У 3.3 `causal_mask` — нижньотрикутна матриця одиниць. Чому саме нижньотрикутна, а не верхньотрикутна? Що б змінилося в поведінці моделі при генерації тексту, якби маски не було?
8. У 3.10 генерація бере чекпоінт із найкращим `val_loss`, а не останній. За яких умов ці два чекпоінти збігаються, а за яких — ні? Подивіться на свою криву навчання (3.9) і скажіть, який випадок у вас.
10. Зменшіть `EVAL_INTERVAL` у 5 разів (наприклад з 250 до 50) і повторно запустіть тренування. Чи стала крива навчання (3.9) помітно гладкішою/інформативнішою? Скільки додаткового часу це коштує (частіші виклики `estimate_loss()` на `EVAL_ITERS` батчах), і чи є сенс робити `EVAL_INTERVAL` ще меншим?
11. У 4.2 той самий промпт дає дуже різний текст при `temperature=0.2` і `temperature=1.4`. Поясніть математично, що робить `temperature` з логітами перед `softmax`, і чому при дуже високій температурі текст стає майже випадковим.
12. За що саме відповідає параметр `top_p`?
13. У 4.2 змініть налаштування `enable_thinking=False` на `True` запустіть повторно клітинку і прокоментуйте зміни у результаті виконання.

### 0.2. Мої відповіді:
1. ...

---
## 1. Свій датасет

Знайдіть **свій** українськомовний текстовий корпус, достатній за обсягом для mini-GPT — наприклад, на [Hugging Face Datasets](https://huggingface.co/datasets?language=language:uk) або на [Kaggle](https://www.kaggle.com/datasets?search=ukrainian). Жанр і епоха — на ваш вибір (новини, художня література, вікіпедія, форуми, судові рішення тощо), головне — одна мова (українська) і достатньо тексту, щоб токенізатор і модель побачили бодай прості закономірності.

⚠️ Якщо датасет великий (сотні МБ тексту і більше) — кодуйте корпус пачками, як у демо 2.2 (`encode_batch` + конвертація в `numpy.int32`), інакше отримаєте помилку нестачі пам'яті (out of memory, OOM).

⚠️ Якщо датасет доступний лише через `streaming=True` (наприклад, через розмір) — `load_dataset(..., streaming=True).take(N)` повертає `IterableDataset`, у якого **нема** `.to_pandas()` і `len()`. Спершу матеріалізуйте вибірку в список, а вже потім — у `DataFrame`:
```python
rows = list(streamed_ds)
poems_df = pd.DataFrame(rows)
```

In [1]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import torch

DATASET_NAME = "podarok/kobza-cleaned-ua"
N_DOCS = 100_000

In [2]:
streamed_ds = load_dataset(
    DATASET_NAME,
    split="train",
    streaming=True
)

rows = list(streamed_ds.take(N_DOCS))
texts_df = pd.DataFrame(rows)

print("Колонки:", texts_df.columns.tolist())
print("Кількість завантажених документів:", len(texts_df))
display(texts_df[["text", "source", "length"]].head(3))

README.md:   0%|          | 0.00/6.33k [00:00<?, ?B/s]

Колонки: ['text', 'source', 'length']
Кількість завантажених документів: 100000


,text,source,length
0,Наркоторговка з Донецька сприйняла арешт як ро...,hplt-2.0,70
1,Жінка пригощала гостей марихуаною © Архів,hplt-2.0,41
2,Коли в квартирі жінки з'явилися співробітники ...,hplt-2.0,120


In [3]:
assert "text" in texts_df.columns, "У датасеті не знайдено колонки text"

documents = (
    texts_df["text"]
    .dropna()
    .astype(str)
    .str.strip()
)

documents = documents[documents.str.len() > 0].tolist()

lengths = [len(text) for text in documents]

print(f"Документів після очищення: {len(documents):,}")
print(f"Середня довжина: {np.mean(lengths):.0f} символів")
print(f"Максимальна довжина: {np.max(lengths):,} символів")
print("\nПриклад тексту:\n")
print(documents[0][:1000])

Документів після очищення: 100,000
Середня довжина: 147 символів
Максимальна довжина: 44,645 символів

Приклад тексту:

Наркоторговка з Донецька сприйняла арешт як розіграш до дня народження


In [4]:
corpus_text = documents


In [5]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

VOCAB_SIZE = 8_000
SPECIAL_TOKENS = ["<unk>", "<eos>"]

tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS
)

In [6]:
tokenizer.train_from_iterator(corpus_text, trainer=trainer)

print("Розмір словника:", tokenizer.get_vocab_size())

tokenizer.save("ukrainian_bpe_tokenizer.json")

Розмір словника: 8000


In [7]:
sample_text = corpus_text[0]
sample_encoding = tokenizer.encode(sample_text)

print("Текст:")
print(sample_text[:300])

print("\nПерші 30 ID токенів:")
print(sample_encoding.ids[:30])

print("\nПерші 30 токенів:")
print(sample_encoding.tokens[:30])

Текст:
Наркоторговка з Донецька сприйняла арешт як розіграш до дня народження

Перші 30 ID токенів:
[355, 301, 299, 835, 235, 211, 262, 210, 4323, 262, 5378, 230, 268, 275, 7055, 329, 343, 6841, 236, 312, 2023, 3197]

Перші 30 токенів:
['ÐĿ', 'Ð°ÑĢ', 'ÐºÐ¾', 'ÑĤÐ¾ÑĢ', 'Ð³', 'Ð¾Ð²', 'ÐºÐ°', 'ĠÐ·', 'ĠÐĶÐ¾Ð½ÐµÑĨÑĮ', 'ÐºÐ°', 'ĠÑģÐ¿ÑĢÐ¸Ð¹', 'Ð½Ñı', 'Ð»Ð°', 'ĠÐ°', 'ÑĢÐµÑĪÑĤ', 'ĠÑıÐº', 'ĠÑĢÐ¾Ð·', 'ÑĸÐ³ÑĢÐ°', 'ÑĪ', 'ĠÐ´Ð¾', 'ĠÐ´Ð½Ñı', 'ĠÐ½Ð°ÑĢÐ¾Ð´Ð¶ÐµÐ½Ð½Ñı']


In [8]:
BATCH_SIZE = 1_000
eos_id = tokenizer.token_to_id("<eos>")

encoded_batches = []

for start in range(0, len(corpus_text), BATCH_SIZE):
    batch_texts = corpus_text[start:start + BATCH_SIZE]

In [9]:
batch_encodings = tokenizer.encode_batch(batch_texts)


In [10]:
batch_arrays = [
        np.asarray(encoding.ids + [eos_id], dtype=np.int32)
        for encoding in batch_encodings
]

encoded_batches.append(np.concatenate(batch_arrays))

if start % (BATCH_SIZE * 20) == 0:
        print(f"Закодовано {min(start + BATCH_SIZE, len(corpus_text)):,} "
              f"із {len(corpus_text):,} документів")

In [11]:
all_token_ids = np.concatenate(encoded_batches)

print("Форма:", all_token_ids.shape)
print("Тип:", all_token_ids.dtype)
print(f"Усього токенів: {len(all_token_ids):,}")
print(f"Пам'ять масиву: {all_token_ids.nbytes / 1024**2:.1f} MB")

Форма: (35886,)
Тип: int32
Усього токенів: 35,886
Пам'ять масиву: 0.1 MB


In [12]:
split_index = int(0.9 * len(all_token_ids))

train_data = torch.from_numpy(all_token_ids[:split_index].astype(np.int64))
val_data = torch.from_numpy(all_token_ids[split_index:].astype(np.int64))

print("train_data:", train_data.shape, train_data.dtype)
print("val_data:", val_data.shape, val_data.dtype)

train_data: torch.Size([32297]) torch.int64
val_data: torch.Size([3589]) torch.int64


Для навчання mini-GPT використано podarok/kobza-cleaned-ua — очищений українськомовний корпус вебтекстів. Датасет завантажено в потоковому режимі, оскільки повна версія велика. Для експерименту використано перші 100 000 непорожніх документів із колонки text. На цих документах навчено BPE-токенізатор зі словником 8 000 токенів. Під час кодування в кінець кожного документа додано спеціальний токен <eos>, який позначає межу між текстами.


---
## 2. Токенізатори: BPE, WordPiece, Unigram

Демо тренує лише BPE. У цій секції ви натренуєте й порівняєте всі три основні алгоритми токенізації на **своєму** тексті.

Матеріал — офіційний [Hugging Face LLM Course, розділ 6](https://huggingface.co/learn/llm-course/en/chapter6/1). Пройдіть і повторіть на своїх даних:

1. **[6.2. Тренування нового токенізатора на основі старого](https://huggingface.co/learn/llm-course/en/chapter6/2)** — найпростіший шлях: `tokenizer.train_new_from_iterator(...)` на основі архітектури готового токенізатора (наприклад, `Qwen/Qwen2.5-0.5B-Instruct`). Зробіть це як розминку перед ручними варіантами нижче.
2. **[6.5. Алгоритм BPE](https://huggingface.co/learn/llm-course/en/chapter6/5)** — повторіть покрокову реалізацію з туторіалу на своєму тексті, повністю вручну або за допомогою бібліотеки `tokenizers`.
3. **[6.6. Алгоритм WordPiece](https://huggingface.co/learn/llm-course/en/chapter6/6)** — так само.
4. **[6.7. Алгоритм Unigram](https://huggingface.co/learn/llm-course/en/chapter6/7)** — так само.

### 1.1. Порівняння

Для однакового `vocab_size` (візьміть, наприклад, 500 — на невеликому зразку тексту більший словник не встигне показати різницю) порівняйте три токенізатори в невеликій таблиці: розмір словника, кількість токенів на вашому зразку, скільки токенів виходить на кількох обраних словах (частотне й рідкісне — як у лекції). Коротко письмово: який алгоритм дав найкоротші послідовності на вашому тексті і чи це узгоджується з тим, що каже туторіал про природу кожного алгоритму?

In [13]:
from transformers import AutoTokenizer

WARMUP_VOCAB_SIZE = 8_000

base_tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct"
)

qwen_ua_tokenizer = base_tokenizer.train_new_from_iterator(
    corpus_text,
    vocab_size=WARMUP_VOCAB_SIZE
)

print("Розмір нового токенізатора:", len(qwen_ua_tokenizer))
print(qwen_ua_tokenizer.tokenize("Українська мова має багату історію."))


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Розмір нового токенізатора: 8008
['Ð£ÐºÑĢÐ°ÑĹÐ½', 'ÑģÑĮÐºÐ°', 'ĠÐ¼Ð¾Ð²Ð°', 'ĠÐ¼Ð°ÑĶ', 'ĠÐ±Ð°Ð³', 'Ð°ÑĤÑĥ', 'ĠÑĸÑģÑĤÐ¾ÑĢÑĸÑİ', '.']


In [14]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
import pandas as pd
import re
from collections import Counter

COMPARISON_VOCAB_SIZE = 500
SPECIAL_TOKENS = ["<unk>", "<eos>"]

In [15]:
tokenizer_train_texts = corpus_text[:10_000]


In [16]:
comparison_text = "\n".join(corpus_text[10_000:11_000])

print(f"Документів для тренування: {len(tokenizer_train_texts):,}")
print(f"Символів у тексті порівняння: {len(comparison_text):,}")

Документів для тренування: 10,000
Символів у тексті порівняння: 106,838


In [17]:
bpe_tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
bpe_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

bpe_trainer = trainers.BpeTrainer(
    vocab_size=COMPARISON_VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS
)

bpe_tokenizer.train_from_iterator(
    tokenizer_train_texts,
    trainer=bpe_trainer
)

In [18]:
wordpiece_tokenizer = Tokenizer(models.WordPiece(unk_token="<unk>"))
wordpiece_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

wordpiece_trainer = trainers.WordPieceTrainer(
    vocab_size=COMPARISON_VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS
)

wordpiece_tokenizer.train_from_iterator(
    tokenizer_train_texts,
    trainer=wordpiece_trainer
)


In [19]:
unigram_tokenizer = Tokenizer(models.Unigram())
unigram_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

unigram_trainer = trainers.UnigramTrainer(
    vocab_size=COMPARISON_VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS,
    unk_token="<unk>"
)

unigram_tokenizer.train_from_iterator(
    tokenizer_train_texts,
    trainer=unigram_trainer
)


In [20]:
word_counts = Counter()

for text in tokenizer_train_texts:
    words = re.findall(r"[А-Яа-яІіЇїЄєҐґ'-]+", text.lower())
    word_counts.update(words)

frequent_word = word_counts.most_common(1)[0][0]

rare_word = next(
    word
    for word, count in word_counts.items()
    if count == 1 and len(word) >= 8
)

print("Частотне слово:", frequent_word, "| частота:", word_counts[frequent_word])
print("Рідкісне слово:", rare_word, "| частота:", word_counts[rare_word])

Частотне слово: і | частота: 3641
Рідкісне слово: наркоторговка | частота: 1


In [21]:
tokenizers_to_compare = {
    "BPE": bpe_tokenizer,
    "WordPiece": wordpiece_tokenizer,
    "Unigram": unigram_tokenizer,
}

results = []

for algorithm, tokenizer in tokenizers_to_compare.items():
    frequent_encoding = tokenizer.encode(frequent_word)
    rare_encoding = tokenizer.encode(rare_word)
    text_encoding = tokenizer.encode(comparison_text)

    results.append({
        "Алгоритм": algorithm,
        "Розмір словника": tokenizer.get_vocab_size(),
        "Токенів у зразку тексту": len(text_encoding.ids),
        "Частотне слово": frequent_word,
        "Токени частотного слова": frequent_encoding.tokens,
        "К-сть токенів (частотне)": len(frequent_encoding.ids),
        "Рідкісне слово": rare_word,
        "Токени рідкісного слова": rare_encoding.tokens,
        "К-сть токенів (рідкісне)": len(rare_encoding.ids),
    })

comparison_df = pd.DataFrame(results)
comparison_df

,Алгоритм,Розмір словника,Токенів у зразку тексту,Частотне слово,Токени частотного слова,К-сть токенів (частотне),Рідкісне слово,Токени рідкісного слова,К-сть токенів (рідкісне)
0,BPE,500,52707,і,[і],1,наркоторговка,"[на, р, ко, тор, го, в, ка]",7
1,WordPiece,500,65346,і,[і],1,наркоторговка,"[на, ##р, ##ко, ##то, ##р, ##го, ##в, ##ка]",8
2,Unigram,500,55442,і,[і],1,наркоторговка,"[на, р, ко, тор, го, в, ка]",7


In [22]:
best_algorithm = comparison_df.loc[
    comparison_df["Токенів у зразку тексту"].idxmin()
]

print(
    f"Найкоротшу послідовність дав {best_algorithm['Алгоритм']}: "
    f"{best_algorithm['Токенів у зразку тексту']:,} токенів."
)


Найкоротшу послідовність дав BPE: 52,707 токенів.


---
## 3. mini-GPT на своїх даних

Повторіть підхід із розділів 2-3 демо-ноутбука на своєму датасеті — не копіюйте код один в один: у вашого датасету інша колонка з текстом, інший обсяг, інший жанр/регістр мови — токенізатор і модель побачать зовсім інші закономірності, ніж на поезії з демо.

### 3.1. Корпус і токенізація

1. Завантажте свій датасет і зберіть `corpus_text` з потрібної колонки (демо 1.2).
2. Натренуйте токенізатор на своєму корпусі (демо 2.1). **Якщо ваш датасет великий** (сотні МБ тексту і більше) — кодуйте корпус пачками, як у демо 2.2 (`encode_batch` + конвертація в `numpy.int32`). Не забудьте `<eos>` між документами (демо 2.2).
3. Спліт train/val (демо 2.3).

### 3.2. Модель: щонайменше 2 варіанти

За зразком розділу 3 демо (`AttentionHead`, `MultiHeadAttention`, `TransformerBlock`, `MiniGPT`) — тренуйте щонайменше 2 варіанти, які відрізняються параметрами: інший `VOCAB_SIZE`, інша `CONTEXT_LEN`, інша кількість шарів/голів (`N_LAYERS`/`N_HEADS`), або інший `MAX_ITERS`. Кожен варіант логуйте в TensorBoard в окрему підпапку (як у демо 3.8), з осмисленою назвою.

### 3.3. Генерація і висновки

Для кожного варіанта — генерація на найкращому за `val_loss` чекпоінті (демо 3.10), той самий промпт для всіх варіантів, щоб порівняння було чесним. Коротко письмово: який варіант дав кращий `val_loss` і чи помітно це за якістю згенерованого тексту? Що саме (зміна словника, глибини мережі, довжини контексту) дало найбільший ефект?

In [23]:
import os
import math
import time
import gc
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.tensorboard import SummaryWriter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [24]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

GPT_VOCAB_SIZE = 8_000
SPECIAL_TOKENS = ["<unk>", "<eos>"]

gpt_tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
gpt_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(
    add_prefix_space=False
)
gpt_tokenizer.decoder = decoders.ByteLevel()

gpt_trainer = trainers.BpeTrainer(
    vocab_size=GPT_VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS
)

gpt_tokenizer.train_from_iterator(corpus_text, trainer=gpt_trainer)

print("Розмір GPT BPE:", gpt_tokenizer.get_vocab_size())

Розмір GPT BPE: 8000


In [25]:
ENCODE_BATCH_SIZE = 1_000
EOS_ID = gpt_tokenizer.token_to_id("<eos>")

token_id_chunks = []

for start in range(0, len(corpus_text), ENCODE_BATCH_SIZE):
    batch = corpus_text[start:start + ENCODE_BATCH_SIZE]
    encodings = gpt_tokenizer.encode_batch(batch)

    for encoding in encodings:
        token_id_chunks.append(
            np.asarray(encoding.ids + [EOS_ID], dtype=np.int32)
        )

token_ids = np.concatenate(token_id_chunks)

data = torch.from_numpy(token_ids.astype(np.int64))

split_index = int(0.9 * len(data))
train_data = data[:split_index]
val_data = data[split_index:]

del token_id_chunks, token_ids
gc.collect()

print("train:", train_data.shape)
print("val:", val_data.shape)
print("Максимальний ID:", int(train_data.max()))


train: torch.Size([4013749])
val: torch.Size([445973])
Максимальний ID: 7999


In [26]:
VOCAB_SIZE = gpt_tokenizer.get_vocab_size()

assert train_data.dtype == torch.long
assert val_data.dtype == torch.long
assert int(train_data.max()) < VOCAB_SIZE

print("VOCAB_SIZE:", VOCAB_SIZE)

VOCAB_SIZE: 8000


In [29]:
import os
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ARTIFACT_DIR = "models"
RUNS_DIR = "runs"

os.makedirs(ARTIFACT_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

print("Device:", device)
print("Checkpoints:", ARTIFACT_DIR)
print("TensorBoard logs:", RUNS_DIR)


Device: cuda
Checkpoints: models
TensorBoard logs: runs


In [30]:
class AttentionHead(nn.Module):
    def __init__(self, embed_dim, head_dim, context_len, dropout):
        super().__init__()
        self.query = nn.Linear(embed_dim, head_dim, bias=False)
        self.key = nn.Linear(embed_dim, head_dim, bias=False)
        self.value = nn.Linear(embed_dim, head_dim, bias=False)
        self.dropout = nn.Dropout(dropout)

        causal_mask = torch.tril(torch.ones(context_len, context_len))
        self.register_buffer("causal_mask", causal_mask)

    def forward(self, x):
        _, T, _ = x.shape

        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        scores = q @ k.transpose(-2, -1) / math.sqrt(k.shape[-1])
        scores = scores.masked_fill(
            self.causal_mask[:T, :T] == 0,
            float("-inf")
        )

        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)

        return weights @ v


class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, n_heads, context_len, dropout):
        super().__init__()

        head_dim = embed_dim // n_heads
        self.heads = nn.ModuleList([
            AttentionHead(embed_dim, head_dim, context_len, dropout)
            for _ in range(n_heads)
        ])

        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.dropout(self.proj(x))


class FeedForward(nn.Module):
    def __init__(self, embed_dim, dropout):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.ReLU(),
            nn.Linear(4 * embed_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, n_heads, context_len, dropout):
        super().__init__()

        self.attention = MultiHeadAttention(
            embed_dim, n_heads, context_len, dropout
        )
        self.feed_forward = FeedForward(embed_dim, dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.feed_forward(self.norm2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        n_heads,
        n_layers,
        context_len,
        dropout
    ):
        super().__init__()

        self.context_len = context_len
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(context_len, embed_dim)

        self.blocks = nn.Sequential(*[
            TransformerBlock(embed_dim, n_heads, context_len, dropout)
            for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size)

    def forward(self, idx, targets=None):
        _, T = idx.shape

        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(
            torch.arange(T, device=idx.device)
        )

        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.shape[-1]),
                targets.reshape(-1)
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=40):
        self.eval()

        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.context_len:]
            logits, _ = self(idx_cond)

            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                values, _ = torch.topk(logits, top_k)
                logits[logits < values[:, [-1]]] = float("-inf")

            probabilities = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probabilities, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)

        return idx

def get_batch(split, context_len, batch_size):
    source = train_data if split == "train" else val_data

    assert len(source) > context_len + 1, (
        f"Замалий {split} для CONTEXT_LEN={context_len}"
    )

    positions = torch.randint(
        len(source) - context_len - 1,
        (batch_size,)
    )

    x = torch.stack([
        source[i:i + context_len]
        for i in positions
    ])

    y = torch.stack([
        source[i + 1:i + context_len + 1]
        for i in positions
    ])

    return x.to(device), y.to(device)

def train_variant(name, config):
    context_len = config["context_len"]
    batch_size = config["batch_size"]
    eval_iters = config["eval_iters"]

    assert config["embed_dim"] % config["n_heads"] == 0

    model = MiniGPT(
        vocab_size=VOCAB_SIZE,
        embed_dim=config["embed_dim"],
        n_heads=config["n_heads"],
        n_layers=config["n_layers"],
        context_len=context_len,
        dropout=config["dropout"],
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["learning_rate"]
    )

    writer = SummaryWriter(
        log_dir=os.path.join(RUNS_DIR, name)
    )

    best_path = os.path.join(ARTIFACT_DIR, f"minigpt_{name}_best.pt")
    first_path = os.path.join(ARTIFACT_DIR, f"minigpt_{name}_first.pt")

    @torch.no_grad()
    def estimate_loss():
        model.eval()
        result = {}

        for split in ("train", "val"):
            losses = torch.zeros(eval_iters)

            for i in range(eval_iters):
                xb, yb = get_batch(split, context_len, batch_size)
                _, loss = model(xb, yb)
                losses[i] = loss.item()

            result[split] = losses.mean().item()

        model.train()
        return result

    history = {
        "iters": [],
        "train_loss": [],
        "val_loss": [],
    }

    best_val_loss = float("inf")
    best_iter = None
    started_at = time.time()

    model.train()

    for iteration in range(config["max_iters"]):
        xb, yb = get_batch("train", context_len, batch_size)
        _, loss = model(xb, yb)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        is_evaluation = (
            iteration % config["eval_interval"] == 0
            or iteration == config["max_iters"] - 1
        )

        if is_evaluation:
            losses = estimate_loss()

            history["iters"].append(iteration)
            history["train_loss"].append(losses["train"])
            history["val_loss"].append(losses["val"])

            writer.add_scalar("Loss/train", losses["train"], iteration)
            writer.add_scalar("Loss/val", losses["val"], iteration)

            print(
                f"{name} | iter {iteration:5d} | "
                f"train={losses['train']:.3f} | "
                f"val={losses['val']:.3f}"
            )

            checkpoint = {
                "model_state": model.state_dict(),
                "config": config,
                "vocab_size": VOCAB_SIZE,
                "best_val_loss": losses["val"],
                "iteration": iteration,
            }

            if iteration == 0:
                torch.save(checkpoint, first_path)

            if losses["val"] < best_val_loss:
                best_val_loss = losses["val"]
                best_iter = iteration
                torch.save(checkpoint, best_path)

    writer.close()

    elapsed_minutes = (time.time() - started_at) / 60
    n_params = sum(p.numel() for p in model.parameters())

    print(f"\n{name}: завершено за {elapsed_minutes:.1f} хв")
    print(f"Параметрів: {n_params:,}")
    print(f"Найкращий val_loss: {best_val_loss:.3f}, iter={best_iter}")

    return {
        "name": name,
        "config": config,
        "best_val_loss": best_val_loss,
        "best_iter": best_iter,
        "best_path": best_path,
        "first_path": first_path,
        "history": history,
        "n_params": n_params,
    }

In [31]:
VARIANT_1 = {
    "context_len": 128,
    "embed_dim": 128,
    "n_heads": 4,
    "n_layers": 4,
    "dropout": 0.1,
    "batch_size": 64,
    "learning_rate": 3e-4,
    "max_iters": 3000,
    "eval_interval": 250,
    "eval_iters": 50,
}

VARIANT_2 = {
    "context_len": 256,
    "embed_dim": 192,
    "n_heads": 4,
    "n_layers": 6,
    "dropout": 0.1,
    "batch_size": 32,
    "learning_rate": 3e-4,
    "max_iters": 3000,
    "eval_interval": 250,
    "eval_iters": 50,
}

run_1 = train_variant("uk_bpe_ctx128_l4", VARIANT_1)

uk_bpe_ctx128_l4 | iter     0 | train=9.107 | val=9.102
uk_bpe_ctx128_l4 | iter   250 | train=7.500 | val=7.479
uk_bpe_ctx128_l4 | iter   500 | train=7.168 | val=7.181
uk_bpe_ctx128_l4 | iter   750 | train=6.795 | val=6.855
uk_bpe_ctx128_l4 | iter  1000 | train=6.520 | val=6.619
uk_bpe_ctx128_l4 | iter  1250 | train=6.293 | val=6.412
uk_bpe_ctx128_l4 | iter  1500 | train=6.098 | val=6.245
uk_bpe_ctx128_l4 | iter  1750 | train=5.920 | val=6.081
uk_bpe_ctx128_l4 | iter  2000 | train=5.773 | val=5.966
uk_bpe_ctx128_l4 | iter  2250 | train=5.631 | val=5.884
uk_bpe_ctx128_l4 | iter  2500 | train=5.552 | val=5.789
uk_bpe_ctx128_l4 | iter  2750 | train=5.473 | val=5.731
uk_bpe_ctx128_l4 | iter  2999 | train=5.403 | val=5.670

uk_bpe_ctx128_l4: завершено за 3.3 хв
Параметрів: 2,864,192
Найкращий val_loss: 5.670, iter=2999


In [32]:
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

run_2 = train_variant("uk_bpe_ctx256_l6", VARIANT_2)

uk_bpe_ctx256_l6 | iter     0 | train=9.041 | val=9.034
uk_bpe_ctx256_l6 | iter   250 | train=7.343 | val=7.344
uk_bpe_ctx256_l6 | iter   500 | train=6.825 | val=6.894
uk_bpe_ctx256_l6 | iter   750 | train=6.454 | val=6.552
uk_bpe_ctx256_l6 | iter  1000 | train=6.121 | val=6.277
uk_bpe_ctx256_l6 | iter  1250 | train=5.865 | val=6.046
uk_bpe_ctx256_l6 | iter  1500 | train=5.651 | val=5.861
uk_bpe_ctx256_l6 | iter  1750 | train=5.515 | val=5.758
uk_bpe_ctx256_l6 | iter  2000 | train=5.373 | val=5.636
uk_bpe_ctx256_l6 | iter  2250 | train=5.300 | val=5.557
uk_bpe_ctx256_l6 | iter  2500 | train=5.170 | val=5.491
uk_bpe_ctx256_l6 | iter  2750 | train=5.130 | val=5.439
uk_bpe_ctx256_l6 | iter  2999 | train=5.062 | val=5.418

uk_bpe_ctx256_l6: завершено за 7.9 хв
Параметрів: 5,795,264
Найкращий val_loss: 5.418, iter=2999


In [34]:
def generate_text(checkpoint_path, prompt,
                  max_new_tokens=80, temperature=0.8, top_k=40):

    checkpoint = torch.load(checkpoint_path, map_location=device)
    config = checkpoint["config"]

    model = MiniGPT(
        vocab_size=checkpoint["vocab_size"],
        embed_dim=config["embed_dim"],
        n_heads=config["n_heads"],
        n_layers=config["n_layers"],
        context_len=config["context_len"],
        dropout=config["dropout"],
    ).to(device)

    model.load_state_dict(checkpoint["model_state"])

    prompt_ids = gpt_tokenizer.encode(prompt).ids
    input_ids = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=device
    )

    output_ids = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k
    )

    return gpt_tokenizer.decode(output_ids[0].tolist())

PROMPT = "Україна"

print("=== Варіант 1: ctx128_l4 ===")
print(generate_text(run_1["best_path"], PROMPT))

print("\n=== Варіант 2: ctx256_l6 ===")
print(generate_text(run_2["best_path"], PROMPT))

=== Варіант 1: ctx128_l4 ===
Україна, до яких не було проаналізах!Угороді про це заявив було це відіграться на тему: це не було за кордон.Нав. Проект, — це в Україну як це не існує за собою, як не віднести, щоб виходять, щоб все це, що вироє ви можете.Нагадаємо, що ви хочетера?Виви

=== Варіант 2: ctx256_l6 ===
Україна (заплана)Враховуєвропейським потенціалень, що це узаголовок, що з таких випадків на початку ХХ століття на місцевіллях, а також в разі до деяких і прими, які наголошується їх використання єдиного рахунку. Вони не має бути використатах.Пашникнення.У США, що ярку з


In [35]:
del train_data, val_data
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

---
## 4. Playground готових моделей

Тур по задачах через `pipeline()` — на основі [HF LLM Course, Chapter 1, §3](https://huggingface.co/learn/llm-course/en/chapter1/3).

_Зверніть увагу_: в межах попередніх завдань Ви могли завантажити памʼять "під завʼязку", тому для безпроблемного виконання наступних секцій вивільніть памʼять:

1. **Спершу збережіть усе потрібне на диск** — чекпоінти mini-GPT (`models/minigpt_v*.pt`), токенізатори, `results.csv`. Секції 4-6 від змінних секцій 1-3 не залежать, тож після цього їх можна сміливо видаляти.
2. **Видаліть великі обʼєкти** — моделі, оптимізатори, тензори `train_data`/`val_data`, корпус і `DataFrame`-и — і зберіть сміття:
   ```python
   import gc, torch

   del model, optimizer, train_data, val_data, corpus_text  # підставте свої імена змінних
   gc.collect()
   if torch.cuda.is_available():
       torch.cuda.empty_cache()
       print(f"GPU: зайнято {torch.cuda.memory_allocated() / 1e9:.2f} GB")
   ```
   Памʼять звільниться лише тоді, коли на обʼєкт не лишиться **жодного** посилання: зверніть увагу на копії в інших змінних, списках і словниках (наприклад, словник із кількома варіантами моделей).
3. **Перевірте, що памʼять справді звільнилась**: індикатор RAM/GPU вгорі праворуч.
4. **У самій секції 4 не тримайте всі pipeline-и одночасно**: кожен `pipeline(...)` завантажує окрему модель. Перевикористовуйте одну змінну (як `classifier` нижче) або робіть `del` + `gc.collect()` після кожної задачі.
5. **Стежте за диском**: моделі кешуються в `~/.cache/huggingface/hub`, і кілька моделей для summarization/translation легко займуть кілька ГБ. Переглянути й почистити кеш: `hf cache ls` / `hf cache rm <id моделі>`.

In [37]:
import pandas as pd

results_df = pd.DataFrame([
    {
        "Варіант": run_1["name"],
        "CONTEXT_LEN": run_1["config"]["context_len"],
        "N_LAYERS": run_1["config"]["n_layers"],
        "EMBED_DIM": run_1["config"]["embed_dim"],
        "Параметрів": run_1["n_params"],
        "Best val_loss": run_1["best_val_loss"],
        "Best iter": run_1["best_iter"],
    },
    {
        "Варіант": run_2["name"],
        "CONTEXT_LEN": run_2["config"]["context_len"],
        "N_LAYERS": run_2["config"]["n_layers"],
        "EMBED_DIM": run_2["config"]["embed_dim"],
        "Параметрів": run_2["n_params"],
        "Best val_loss": run_2["best_val_loss"],
        "Best iter": run_2["best_iter"],
    },
])

results_df.to_csv("results.csv", index=False)
results_df

,Варіант,CONTEXT_LEN,N_LAYERS,EMBED_DIM,Параметрів,Best val_loss,Best iter
0,uk_bpe_ctx128_l4,128,4,128,2864192,5.669691,2999
1,uk_bpe_ctx256_l6,256,6,192,5795264,5.418180,2999


In [38]:
gpt_tokenizer.save(os.path.join(ARTIFACT_DIR, "uk_bpe_tokenizer.json"))

results_df.to_csv("results.csv", index=False)

print("Збережено:")
print("-", ARTIFACT_DIR)
print("-", "results.csv")

Збережено:
- models
- results.csv


In [39]:
import gc
import torch

objects_to_remove = [
    "train_data",
    "val_data",
    "data",
    "corpus_text",
    "documents",
    "texts_df",
    "all_token_ids",
    "token_id_chunks",
    "tokenizer_train_texts",
    "comparison_text",
    "gpt_tokenizer",
    "qwen_ua_tokenizer",
]

for object_name in objects_to_remove:
    if object_name in globals():
        del globals()[object_name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU зайнято: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


GPU зайнято: 0.02 GB


In [40]:
del pipeline_variable
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

NameError: name 'pipeline_variable' is not defined

### 4.0. Запустіть приклади виконання різних NLP задач

In [41]:
!pip install "transformers[sentencepiece]==5.18.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 48.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.1
    Uninstalling transformers-5.20.1:
      Successfully uninstalled transformers-5.20.1


In [42]:
# Sentiment analysis
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
print(classifier("I've been waiting for a HuggingFace course my whole life."))
print(classifier(
    ["I've been waiting for a HuggingFace course my whole life.", "I hate this so much!"]
))

ImportError: cannot import name '_validate_tp_plan_styles' from 'transformers.distributed.tensor_parallel' (/usr/local/lib/python3.13/dist-packages/transformers/distributed/tensor_parallel.py)

In [ ]:
# Zero-shot classification
from transformers import pipeline

classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

In [ ]:
# Text generation
from transformers import pipeline

generator = pipeline("text-generation")
print(generator("In this course, we will teach you how to"))

generator = pipeline("text-generation", model="distilgpt2")
print(generator(
    "In this course, we will teach you how to",
    max_length=30,
    num_return_sequences=2,
))

In [ ]:
# Fill-mask
from transformers import pipeline

unmasker = pipeline("fill-mask")
unmasker("This course will teach you all about <mask> models.", top_k=2)

In [ ]:
# Named entity recognition
from transformers import pipeline

ner = pipeline("ner", aggregation_strategy="simple")
ner("My name is Sylvain and I work at Hugging Face in Brooklyn.")

In [43]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Question answering
from transformers import pipeline

question_answerer = pipeline("table-question-answering")
question_answerer(
    query="How many stars does the transformers repository have?",
    table={"Repository":["Transformers","Datasets","Tokenizers"],"Stars":["36542","4512","3934"],"Contributors":["651","77","34"],"Programming language":["Python","Python","Rust, Python and NodeJS"]},
)

### 4.1. Свої приклади

Повторіть **усі 6 задач** вище (sentiment-analysis, zero-shot-classification, text-generation, fill-mask, NER, table-question-answering), але замість дефолтних англійських прикладів — придумайте й використайте **свої українськомовні приклади** для кожної задачі (якщо потрібно, підберіть відповідні моделі зі знанням української мови). Для виконання цього завдання постворюйте клітинки нижче.

---
## 5. Порівняння моделей

### 5.1. Оберіть задачу й метрику

Візьміть **одну** задачу зі списку нижче:

| Задача | Метрика | Що потрібно підготувати | Метрика в `evaluate` |
|---|---|---|---|
| sentiment-analysis | Accuracy / F1 | 15-20 своїх речень + правильна мітка (positive/negative) | [`accuracy`](https://huggingface.co/spaces/evaluate-metric/accuracy), [`f1`](https://huggingface.co/spaces/evaluate-metric/f1) |
| zero-shot-classification | Accuracy | те саме + фіксований список `candidate_labels` | [`accuracy`](https://huggingface.co/spaces/evaluate-metric/accuracy) |
| NER | F1 (entity-level) | кілька речень + список правильних сутностей (текст + тип) | [`seqeval`](https://huggingface.co/spaces/evaluate-metric/seqeval) |

Усі метрики рахуються через бібліотеку `evaluate` (той самий автор, що й `transformers`/`datasets`):
```python
import evaluate

metric = evaluate.load("rouge")  # або "accuracy", "f1", "seqeval", "squad", "sacrebleu"
metric.compute(predictions=[...], references=[...])
```
Формат `predictions`/`references` трохи відрізняється між метриками — дивіться приклад використання на сторінці кожної метрики (посилання в таблиці) або [HF Evaluate — Quick tour](https://huggingface.co/docs/evaluate/a_quick_tour).

### 5.2. Порівняйте моделі

На обраній задачі порівняйте щонайменше 2 різні моделі на однакових вхідних прикладах — не лише значення метрики, а й швидкість інференсу та розмір моделі (кількість параметрів).

### 5.3. Поекспериментуйте з налаштуваннями

Візьміть модель, що показала кращий результат, і поекспериментуйте з її налаштуваннями (temperature/top-p/top-k для генеративних задач, поріг впевненості для класифікаційних задач тощо) — щонайменше 3 зміни, по одній за раз, щоб зрозуміти, що саме впливає на метрику. Зведіть усі спроби (модель × налаштування → значення метрики) в невелику таблицю.

Коротко письмово: яка модель і яка комбінація налаштувань дала найкращий результат за обраною метрикою, і наскільки суттєво налаштування вплинули порівняно з вибором моделі?

---
## 6. Творчий застосунок

Побудуйте невеликий Gradio-застосунок на основі будь-якої готової моделі з розділів 4-5 — тема й формат повністю на ваш вибір, заохочується творчість і щось не зовсім очевидне (не просто "ще один чат-бот"). Це може бути й генеративна LLM через `transformers` (`AutoModelForCausalLM`/`AutoTokenizer` або `pipeline("text-generation", ...)`, той самий патерн, що й у демо 4.1-4.2 на Qwen) — `text-generation` уже входить до обов'язкових 8 задач розділу 4, тож це не вихід за межі завдання. Головне — щоб застосунок реально запускався і демонстрував конкретну, зрозумілу користувацьку цінність.

---
### Як здавати лабу

У [форму](https://forms.gle/Z1gxmsDs656jqsDGA) здати посилання на **публічний** репозиторій, який містить:

1. **Демо-ноутбук з усіма виконаними пунктами**. Має запускатися зверху до низу (Run All) в будь-якому середовищі без помилок.
2. **Цей ноутбук з усіма виконаними пунктами**. Має запускатися зверху до низу (Run All) в будь-якому середовищі без помилок.
3. **Файли всіх натренованих варіантів mini-GPT** (3.2) — кожен окремим `.pt` файлом.
4. **Логи TensorBoard** — папка `runs/` з усіма прогонами (і mini-GPT, і, за бажання, логи порівняння в розділі 5).
5. **Таблицю результатів** — `results.csv` з порівнянням варіантів mini-GPT (val_loss) і моделей з розділу 5.
6. Структура файлів на репозиторії, **точно дотримуйтесь, інакше не зарахується лаба**:
```
/lab3/transformers_demo.ipynb
/lab3/transformers_task.ipynb
/lab3/models/minigpt_v1.pt
/lab3/models/minigpt_v2.pt
/lab3/models/...        (усі інші варіанти з 3.2)
/lab3/runs/*
/lab3/results.csv
```